### Sleep fragmentation vs next-day heart rate

This notebook reads `sleep.csv` and `heart_rate.csv`, and specifically explores whether sleep fragmentation (e.g., awake time during sleep) is associated with next-day heart rate metrics.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# heart_rate.csv sample:
# "user_code","datetime","heart_rate","is_resting"
# "007b8190cf","2020-04-26 04:49:25",70,0
# "01bad5a519","2020-04-23 06:21:03",74,0

# sleep.csv sample:
# "user_code","day","sleep_begin","sleep_end","sleep_duration","sleep_awake_duration","sleep_rem_duration","sleep_light_duration","sleep_deep_duration","pulse_min","pulse_max","pulse_average"
# "0d297d2410","2019-12-31","2019-12-31 07:50:32","2019-12-31 08:45:22","3290.000","","","","","","",""
# "0d297d2410","2020-01-01","2020-01-01 04:13:41","2020-01-01 09:45:02","19881.000","","","","","","",""


sleep = pd.read_csv('sleep.csv',parse_dates=['day', 'sleep_begin', 'sleep_end'])
hr = pd.read_csv('heart_rate.csv',parse_dates=['datetime'])

sleep.head()
hr.head()

,user_code,datetime,heart_rate,is_resting
0,007b8190cf,2020-04-26 04:49:25,70,0
1,01bad5a519,2020-04-23 06:21:03,74,0
2,01bad5a519,2020-04-23 09:46:01,82,0
3,01bad5a519,2020-04-23 14:05:06,90,0
4,01bad5a519,2020-04-24 03:41:18,72,0


In [12]:

sleep['sleep_duration_hours'] = sleep['sleep_duration'] / 3600.0
sleep['awake_duration_min'] = sleep['sleep_awake_duration'] / 60.0

# new feature: fragmentation ratio = sleep_awake_duration / sleep_duration
sleep['fragmentation_ratio'] = np.where(
    (sleep['sleep_duration'] > 0) & sleep['sleep_awake_duration'].notna(),
    sleep['sleep_awake_duration'] / sleep['sleep_duration'],
    np.nan
)

sleep['next_day'] = (sleep['day'] + pd.Timedelta(days=1)).dt.date
hr['date'] = hr['datetime'].dt.date

print(sleep[['user_code','day','sleep_duration_hours','awake_duration_min','fragmentation_ratio']].describe(include='all'))
sleep.head()

         user_code                            day  sleep_duration_hours  \
count          425                            425            425.000000   
unique          10                            NaN                   NaN   
top     6be5033971                            NaN                   NaN   
freq           108                            NaN                   NaN   
mean           NaN  2020-03-10 09:15:40.235293952              7.138814   
min            NaN            2019-12-31 00:00:00              0.258333   
25%            NaN            2020-01-30 00:00:00              6.071389   
50%            NaN            2020-02-29 00:00:00              7.233611   
75%            NaN            2020-04-22 00:00:00              8.500000   
max            NaN            2020-06-17 00:00:00             13.183333   
std            NaN                            NaN              2.159390   

        awake_duration_min  fragmentation_ratio  
count             9.000000             9.000000  

,user_code,day,sleep_begin,sleep_end,sleep_duration,sleep_awake_duration,sleep_rem_duration,sleep_light_duration,sleep_deep_duration,pulse_min,pulse_max,pulse_average,fragmentation_ratio,next_day,sleep_duration_hours,awake_duration_min
0,0d297d2410,2019-12-31,2019-12-31 07:50:32,2019-12-31 08:45:22,3290.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-01,0.913889,NaN
1,0d297d2410,2020-01-01,2020-01-01 04:13:41,2020-01-01 09:45:02,19881.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-02,5.522500,NaN
2,0d297d2410,2020-01-02,2020-01-02 02:14:52,2020-01-02 08:06:00,21068.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-03,5.852222,NaN
3,0d297d2410,2020-01-03,2020-01-03 00:10:00,2020-01-03 08:45:10,30910.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-04,8.586111,NaN
4,0d297d2410,2020-01-04,2020-01-04 01:27:25,2020-01-04 08:52:20,26695.0,NaN,NaN,21480.0,NaN,55.0,95.0,72.5,NaN,2020-01-05,7.415278,NaN


### Not able to finish it, but the goal was to find the correlation between a poor night of sleep due to sleep fragmentation with heart-rate variability with the next day. I have no idea if there would be any correlation so I was curious to explore this relationship.